In [ ]:
############################################### 无出图版：gm=Ca=400 的 gm' ##################################

import win32com.client as win32
import pythoncom
import os, csv, time
import numpy as np
import pandas as pd
from glob import glob

# ==== Excel COM 缓存修复 ====
win32.gencache.is_readonly = True
win32.gencache.EnsureModule = lambda *args, **kwargs: None

# ==== 常量 ====
CSV_HEADER = ["File", "Vcmax", "J", "Rd", "GammaS",
              "Vcmax25", "J25", "R2", "RMSE", "SSE", "gm"] 
TEMPLATE_PATH = r"C:/Users/DELL/Desktop/Vcmax/nph14260-sup-0002-methodss211.xlsm"
SHEET_NAME = "nph14260-supp-002"

# ==== 工具函数 ====
def kill_excel_process():
    os.system('taskkill /f /im excel.exe >nul 2>&1')
    time.sleep(1)

def write_column_data(ws, start_cell, data, debug_name=""):
    try:
        if not isinstance(data, (list, np.ndarray)):
            raise ValueError("data 必须是列表或 numpy 数组")
        col = ''.join([c for c in start_cell if c.isalpha()])
        row0 = int(''.join([c for c in start_cell if c.isdigit()]))
        end_row = row0 + len(data) - 1
        rng = f"{col}{row0}:{col}{end_row}"
        ws.Range(rng).Value = np.array(data, dtype=float).reshape(-1, 1)
    except Exception as e:
        print(f"[Excel写入错误] {debug_name} - {e}")
        raise

def write_result_to_csv(data, filename, is_new_run=False):
    try:
        if is_new_run and os.path.exists(filename):
            os.remove(filename)
            print(f"♻️ 已清除旧文件 {filename}")
        file_exists = os.path.exists(filename)
        with open(filename, 'a', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(CSV_HEADER)
            w.writerow([data.get(k, "") for k in CSV_HEADER])
    except Exception as e:
        print(f"[CSV写入错误] {e}")

def safe_excel_operation(func):
    def wrapper(*args, **kwargs):
        excel = None
        pythoncom.CoInitialize()
        try:
            kill_excel_process()
            excel = win32.dynamic.Dispatch("Excel.Application")
            excel.Visible = False
            excel.DisplayAlerts = False
            return func(excel, *args, **kwargs)
        except Exception as e:
            print(f"[Excel操作错误] {e}")
            return None
        finally:
            if excel:
                try:
                    excel.Quit()
                except:
                    pass
            pythoncom.CoUninitialize()
            kill_excel_process()
    return wrapper

# ==== 主求解 ====
@safe_excel_operation
def run_excel_solver(excel, tpl_path, T, A, Ci, Pci, PARi, PhiPS2, P,
                     filename=None, Ca=None):
    wb = None
    try:
        wb = excel.Workbooks.Open(os.path.abspath(tpl_path))
        ws = wb.Sheets(SHEET_NAME)

        # 清空旧数据区域
        ws.Range("B20:B45,C20:C45,D20:D45,E20:E45,F20:F45").ClearContents()

        # 写入平均温度/气压
        ws.Range("B12").Value = round(float(np.mean(T)), 2)
        ws.Range("B13").Value = round(float(np.mean(P)), 2)

        # 写入列数据
        write_column_data(ws, "B20", A, "A")
        write_column_data(ws, "C20", Ci, "Ci")
        write_column_data(ws, "D20", Pci, "Pci")
        write_column_data(ws, "E20", PARi, "PARi")
        write_column_data(ws, "F20", PhiPS2, "PhiPS2")

        # Solver
        solver_path = r"C:\Program Files\Microsoft Office\root\Office16\Library\SOLVER\SOLVER.XLAM"
        solver_prefix = f"'{solver_path}'!"

        excel.Run(f"{solver_prefix}SolverReset")
        excel.CalculateFullRebuild()

        # 
        excel.Run(f"{solver_prefix}SolverOk", "$X$22", 2, "0", "$P$20,$P$21,$P$23")
        excel.Run(f"{solver_prefix}SolverAdd", "$P$20", 3, 30)    # Vcmax >= 30
        excel.Run(f"{solver_prefix}SolverAdd", "$P$21", 3, 30)    # J >= 30
        excel.Run(f"{solver_prefix}SolverAdd", "$P$23", 1, 0.55)  # Tau <= 0.55
        excel.Run(f"{solver_prefix}SolverAdd", "$P$24", 1, 400)   # CCtr <= 400
        excel.Run(f"{solver_prefix}SolverAdd", "$P$24", 3, "$P$40")
        excel.Run(f"{solver_prefix}SolverAdd", "$P$40", 1, 70)

        excel.Run(f"{solver_prefix}SolverOptions", 100, 100, 1e-6, True)
        excel.Run(f"{solver_prefix}SolverSolve", True)
        excel.CalculateFullRebuild()

        # ----- 提取序列 -----
        n = len(Ci) + (20 - 1)
        gm_values = ws.Range(f"H20:H{n}").Value
        gm_list = [float(v[0]) for v in gm_values if v and isinstance(v, (list, tuple))]

        # ----- 统计量 -----
        sse = float(ws.Range("X22").Value)
        rmse = float(np.sqrt(sse / len(T)))
        tss = float(np.sum((A - np.mean(A)) ** 2))
        r2 = float(1 - (sse / tss)) if tss != 0 else float('nan')

        def _gm_at_ca(ca_seq, gm_seq, x=400.0):
            try:
                ca = np.asarray(ca_seq, dtype=float)
                gm = np.asarray(gm_seq, dtype=float)
                m = np.isfinite(ca) & np.isfinite(gm)
                ca, gm = ca[m], gm[m]
                if ca.size < 2:
                    return float('nan')
                order = np.argsort(ca)
                ca, gm = ca[order], gm[order]
                if ca.min() <= x <= ca.max():
                    return float(np.interp(x, ca, gm))
                k, b = np.polyfit(ca, gm, 1)
                return float(k * x + b)
            except Exception:
                return float('nan')

        gm_at = _gm_at_ca(Ca[:len(gm_list)] if Ca is not None else [], gm_list)
        GammaS = np.mean((np.exp(11.187 - 24.46 / (8.314e-3 * (T + 273.15)))) / (P * 0.001))

        result = {
            "File": filename,
            "Vcmax": ws.Range("P20").Value,
            "J": ws.Range("P21").Value,
            "Rd": ws.Range("P22").Value,
            "GammaS": GammaS,
            "Vcmax25": ws.Range("Q20").Value,
            "J25": ws.Range("Q21").Value,           
            "R2": r2,
            "RMSE": rmse,
            "SSE": sse,            
            "gm": f"{gm_at:.4f}" if np.isfinite(gm_at) else ""
        }

        return result

    except Exception as e:
        print(f"[Solver执行失败] {e}")
        return None
    finally:
        if wb:
            wb.Close(False)

# ==== 单文件处理 ====
def process_aci_file(fpath, csv_out):
    try:
        df = pd.read_excel(fpath, skiprows=14, engine='openpyxl')
        print(f"📄 处理文件: {os.path.basename(fpath)}")

        cols = ['A', 'Tleaf', 'Pa', 'Ci', 'Pci', 'Qin', 'PhiPS2', 'Ca']
        data = df.iloc[1:][cols].dropna()

        A = data['A'].astype(np.float32).values
        T = data['Tleaf'].astype(np.float32).values
        P = data['Pa'].astype(np.float32).values
        Ci = data['Ci'].astype(np.float32).values
        Pci = data['Pci'].astype(np.float32).values
        PARi = data['Qin'].astype(np.float32).values
        PhiPS2 = data['PhiPS2'].astype(np.float32).values
        Ca = data['Ca'].astype(np.float32).values

        res = run_excel_solver(
            TEMPLATE_PATH, T, A, Ci, Pci, PARi, PhiPS2, P,
            filename=os.path.basename(fpath),
            Ca=Ca
        )

        if res:
            write_result_to_csv(res, csv_out)
            return True
        return False

    except Exception as e:
        print(f"❌ 处理失败: {e}")
        return False

# ==== 批处理 ====
def batch_process():
    try:
        #folders = ["winter wheat", "红三叶","小麦","黑麦"]  
        #folders = ['raw20190627SY','raw20190709SY','raw20190728SY','raw20190803SY','raw20190810SY','raw20190818SY','raw20190823SY','raw20190907SY','raw20200615SY']
        #folders = ["0615", "0625","0706","0717","0724","0807","0815","0823"]   ##### 大豆02
        folders = ['raw20190627SY']

        for n in folders:
            inp = f"D:/博士学习资料/根据Vcmax25计算A/大豆01/{n}"
            outd = f"D:/博士学习资料/Result/大豆_01/0.Moualeu-Ngangue_2017_{n}"

            os.makedirs(outd, exist_ok=True)
            outcsv = os.path.join(outd, "result.csv")

            files = sorted([f for f in glob(f"{inp}/*.xlsx") if not os.path.basename(f).startswith("~$")])

            if os.path.exists(outcsv):
                os.remove(outcsv)

            for f in files:
                process_aci_file(f, outcsv)

        print("🎉 全部处理完成")

    finally:
        kill_excel_process()

# ==== 入口 ====
if __name__ == "__main__":
    try:
        pythoncom.CoInitialize()
        excel = win32.dynamic.Dispatch("Excel.Application")
        print(f"✔ Excel {excel.Version} 连接成功")
        excel.Quit()
    except Exception as e:
        print(f"× Excel连接失败: {e}")
        exit(1)
    finally:
        pythoncom.CoUninitialize()
        kill_excel_process()

    batch_process()


✔ Excel 16.0 连接成功
📄 处理文件: 2019-06-27-ACi-top2.xlsx
📄 处理文件: 2019-06-27-ACi-top3.xlsx
📄 处理文件: 2019-06-27-ACi-top4.xlsx
📄 处理文件: 2019-06-27-ACi-top5.xlsx
🎉 全部处理完成
